In [ ]:
BACKEND = "local"   # "local" hoặc "gemini"
N = 100           # số câu hỏi (đặt 5 để chạy thử nhanh)

## 1. Cài đặt và lấy mã

In [ ]:
!git clone -q https://github.com/FluteInTheRain/evograph-reproduction.git
%cd evograph-reproduction
!pip install -q -r requirements.txt gdown

## 2. Đăng nhập W&B

In [ ]:
import wandb
wandb.login()

## 3. Dữ liệu MetaQA
Hai tệp `kb.txt` và `2hop_qa_test.txt` (MetaQA, https://github.com/yuyuz/MetaQA) đã nằm sẵn trong thư mục `data/` của repo.

In [ ]:
import os
assert os.path.exists("data/kb.txt") and os.path.exists("data/2hop_qa_test.txt"), "Thiếu dữ liệu: git pull để lấy thư mục data/"
!wc -l data/kb.txt data/2hop_qa_test.txt

## 4. Chọn LLM

In [ ]:
import subprocess, time, os, requests
if BACKEND == "local":
    !pip install -q vllm
    server = subprocess.Popen(
        ["python", "-m", "vllm.entrypoints.openai.api_server",
         "--model", "Qwen/Qwen2.5-7B-Instruct-AWQ", "--quantization", "awq", "--dtype", "half",
         "--max-model-len", "6144", "--gpu-memory-utilization", "0.9", "--port", "8000"],
        stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(120):
        try:
            requests.get("http://localhost:8000/v1/models", timeout=2); break
        except Exception:
            time.sleep(5)
    os.environ["LLM_KEY"] = "EMPTY"
    LLM = "--model Qwen/Qwen2.5-7B-Instruct-AWQ --base_url http://localhost:8000/v1 --api_key_env LLM_KEY"
else:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    LLM = "--model gemini-2.5-flash-lite --base_url https://generativelanguage.googleapis.com/v1beta/openai/ --api_key_env GEMINI_API_KEY"
print(LLM)

## 5. Chạy thử nhanh (5 câu)

In [ ]:
!python run.py --kb data/kb.txt --qa data/2hop_qa_test.txt --variant full --n 5 --wandb_group smoke-test $LLM

## 6. Bốn cấu hình chính

In [ ]:
for v in ["static", "web", "edit", "full"]:
    !python run.py --kb data/kb.txt --qa data/2hop_qa_test.txt --variant $v --n $N --wandb_group main-ablation $LLM

## 7. Ablation theo mức khuyết của KG

In [ ]:
for d in [0.0, 0.25, 0.5, 0.75]:
    for v in ["static", "full"]:
        !python run.py --kb data/kb.txt --qa data/2hop_qa_test.txt --variant $v --n $N --drop $d --wandb_group ablation-drop $LLM

## 8. Ablation theo độ nhiễu của web

In [ ]:
for p in [0.0, 0.1, 0.3, 0.5]:
    !python run.py --kb data/kb.txt --qa data/2hop_qa_test.txt --variant full --n $N --web_noise $p --wandb_group ablation-webnoise $LLM